## WAD-DIV Example Usage

### Load Feature Embeddings

First, we load precomputed feature embeddings stored in `.npz` files.  
The synthetic dataset contains embeddings generated by the generative model, while the real dataset contains embeddings extracted from the training data.

Both datasets must:
- Be computed using the same feature extractor (e.g., MedicalNet 3D),
- Have identical dimensionality,
- Use identical preprocessing.

These embeddings serve as the input to the WAD-Div metric.

In [25]:
import numpy as np

# Synthetic dataset
data = np.load("example_embeddings/luna_1unique_medicalnet.npz")

# Real dataset
data_real = np.load("example_embeddings/luna_100unique_medicalnet.npz")

synthetic_features = data["Embeddings"]
real_features = data_real["Embeddings"]

print("Synthetic shape:", synthetic_features.shape)
print("Real shape:", real_features.shape)

Synthetic shape: (100, 2048)
Real shape: (100, 2048)


### Initialize the WAD-Div Metric

Here, we instantiate the `WADDiv` class.

The metric computes Wasserstein distance between k-nearest-neighbor (kNN) distance distributions in feature space.

Parameters:
- `distance_metric`: Distance metric used for kNN computation (default: Euclidean).

At this stage, no normalization anchor has been fitted yet.

In [26]:
from waddiv import WADDiv  

metric = WADDiv(distance_metric="euclidean")

### Compute WAD-Div with Zero Reference (Raw)

This setting measures structural diversity relative to complete mode collapse.

The reference distribution is a Dirac delta at zero distance, representing identical samples.

Interpretation:
- Higher values indicate greater diversity.
- Values close to zero indicate low diversity or collapse.

This configuration is useful for directly comparing generative models without requiring a real reference dataset.

In [27]:
result_zero = metric.compute(
    features=synthetic_features,
    k=3,
    reference="zero",
    normalize=False
)

print("Zero reference (raw):", result_zero["wad_div"])

Zero reference (raw): 0.0033982101527078


### Compute WAD-Div with Exponential Reference (Raw)

In this setting, diversity is measured relative to a soft low-variability baseline modeled as an exponential distribution.

The exponential reference is fitted by matching a chosen percentile of the observed kNN distance distribution.

Interpretation:
- Higher values indicate stronger deviation from limited variability.
- Provides a smoother baseline than the zero reference.

In [28]:
result_exp = metric.compute(
    features=synthetic_features,
    k=3,
    reference="exponential",
    ref_percentile=95,
    ref_prob=0.95,
    random_state=42,
    normalize=False
)

print("Exponential reference (raw):", result_exp["wad_div"])

Exponential reference (raw): 0.0016527072833015274


### Compute WAD-Div with Empirical Reference (Real Dataset)

Here, the synthetic dataset is compared directly to the real dataset.

The reference distribution is the kNN distance distribution computed from real data.

Interpretation:
- Lower values indicate closer diversity to real data.
- Higher values indicate stronger deviation from real data diversity.

This configuration evaluates how well a generative model reproduces real-world structural diversity.

In [29]:
result_emp = metric.compute(
    features=synthetic_features,
    k=3,
    reference="empirical",
    ref_features=real_features,
    normalize=False
)

print("Empirical reference (raw):", result_emp["wad_div"])

Empirical reference (raw): 0.11888747920686214


### Fit Maximum Diversity Anchor for Normalization

To compute a **normalized WAD-Div score** in [0, 1], we first determine the diversity of a **maximally diverse dataset**. In generative modeling settings, this is typically the **real training dataset**.  

This value is stored internally as `W_max` and serves as the reference for normalization.  

**Important:** The same metric configuration (k, reference type, and embedding space) must be used for both fitting and evaluation.  

#### Normalization Methods

1. **Linear (default):**  
Maps the raw Wasserstein distance to `[0, 1]` using:  


$W_{\text{norm}} = \text{clip}\Big(\frac{W}{W_{\max}}, 0, 1\Big)$


- `0` → minimal diversity (mode collapse)  
- `~1` → diversity comparable to the reference dataset  
- Intuitive and robust for single synthetic datasets  

2. **Paper formula (optional):**  
The original method from the BVM26 paper:  

$
W_{\text{norm}} = \frac{W}{W + s}, \quad s = W_{\max} \frac{1-t}{t}, \quad t \approx 0.99
$

- Intended for comparisons across multiple datasets  
- Can **inflate scores** when evaluating a single synthetic dataset  

This normalized score allows for **easy comparison across models or experiments**, while providing flexibility to use either the practical linear scaling or the paper’s original formula.

In [32]:
# Compute W_max on the real dataset (maximally diverse reference)
W_max = metric.fit_max_reference(
    features=real_features,
    k=3,
    reference="zero"
)

print("W_max (real dataset):", W_max)

# Normalization method is "linear" by default; can also set normalization_method="paper"
result_norm = metric.compute(
    features=synthetic_features,
    k=3,
    reference="zero",
    normalize=True
)

print("Zero reference (normalized):", result_norm["wad_div_norm"])

W_max (real dataset): 0.12228568935956993
Zero reference (normalized): 0.027789107380469292


### Role of k in WAD-Div

The parameter `k` controls the neighborhood size used to compute local distances:

- **Small k (e.g., 1–5):** Emphasizes local diversity; detects fine-scale differences and mode collapse.
- **Large k (e.g., 20+):** Captures global or structural diversity; measures overall coverage of the feature space.
- **Default recommendation:** Use k=3 for general comparisons. Optionally sweep k to explore diversity at multiple scales.
